In [0]:
%run ../common/load_json

In [0]:
%run ../common/write_to_delta_tables

In [0]:
import json
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, DateType
from pyspark.sql.functions import current_timestamp 
from pyspark.sql.functions import input_file_name
from pyspark.sql.functions import col

In [0]:
raw_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
dir_name = "/".join(raw_path.split("/")[:-2])
config_file_path = f"/Workspace{dir_name}/process_config.json"
# dbutils.widgets.removeAll()
dbutils.widgets.text("config_file_path", config_file_path)
temp = dbutils.widgets.get("config_file_path")
# print(temp)

In [0]:
config = read_json_from_file_path(dbutils.widgets.get("config_file_path"))



In [0]:
#Schema for customers csv files
customers_schema = StructType([
    StructField("customer_id",StringType(),True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("city", StringType(), True),
    StructField("pincode", IntegerType(), True),
    StructField("customer_segment", StringType(), True),
    StructField("acquisition_channel", StringType(), True),
    StructField("registration_date", DateType(), True),
    StructField("is_active", StringType(), True),
    StructField("_data_quality_issue", StringType(), True)
    
    

])

#Schema for dark_stores csv files

dark_stores_schema = StructType([
    StructField("store_id", StringType(), True),
    StructField("store_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("zone", StringType(), True),
    StructField("address", StringType(), True),
    StructField("pincode", IntegerType(), True),
    StructField("capacity_sqft", IntegerType(), True),
    StructField("max_orders_per_day", IntegerType(), True),
    StructField("operational_since", DateType(), True),
    StructField("store_status", StringType(), True),
    StructField("delivery_radius_km", DoubleType(), True),
    StructField("_data_quality_issue", StringType(), True)
    
])

#Schema for deliveries csv files

deliveries_schema = StructType([
    StructField("delivery_id", StringType(), True),
    StructField("order_id", StringType(), True),
    StructField("delivery_partner_id", StringType(), True),
    StructField("store_id", StringType(), True),
    StructField("pickup_timestamp", TimestampType(), True),
    StructField("delivered_timestamp", TimestampType(), True),
    StructField("delivery_minutes", DoubleType(), True),
    StructField("is_sla_breached", StringType(), True),
    StructField("delivery_status", StringType(), True),
    StructField("_data_quality_issue", StringType(), True)
    
])

#Schema for inventory csv files

inventory_schema = StructType([
    StructField("inventory_id", StringType(), True),
    StructField("store_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("snapshot_date", DateType(), True),
    StructField("units_available", IntegerType(), True),
    StructField("units_sold", IntegerType(), True),
    StructField("reorder_level", IntegerType(), True),
    StructField("is_out_of_stock", StringType(), True),
    StructField("is_weekend", StringType(), True),
    StructField("_data_quality_issue", StringType(), True)
    
])

#Schema for orders csv files

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("store_id", StringType(), True),
    StructField("order_timestamp", StringType(), True),
    StructField("order_items", StringType(), True),
    StructField("order_amount", DoubleType(), True),
    StructField("delivery_charge", DoubleType(), True),
    StructField("final_amount", DoubleType(), True),
    StructField("payment_method", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("is_weekend", StringType(), True),
    StructField("_data_quality_issue", StringType(), True)
])

In [0]:
# Tables Path
customers_table = f"{config['raw_catalog_name']}.{config['raw_schema_name']}.customers"
dark_stores_table = f"{config['raw_catalog_name']}.{config['raw_schema_name']}.dark_stores"
deliveries_table = f"{config['raw_catalog_name']}.{config['raw_schema_name']}.deliveries"
inventory_table = f"{config['raw_catalog_name']}.{config['raw_schema_name']}.inventory"
orders_table = f"{config['raw_catalog_name']}.{config['raw_schema_name']}.orders"


# Function call
read_and_write(customers_schema, customers_table, config["source_files"]["customers"])
read_and_write(dark_stores_schema, dark_stores_table, config["source_files"]["dark_stores"])
read_and_write(deliveries_schema, deliveries_table, config["source_files"]["deliveries"])
read_and_write(inventory_schema, inventory_table, config["source_files"]["inventory"])
read_and_write(orders_schema, orders_table, config["source_files"]["orders"])



In [0]:
%skip
# Customers table

customers_df.write.mode("append").saveAsTable(f"{config['raw_catalog_name']}.{config['raw_schema_name']}.customers")

# Dark Stores table

dark_stores_df.write.mode("append").saveAsTable(f"{config['raw_catalog_name']}.{config['raw_schema_name']}.dark_stores")

# Deliveries table

deliveries_df.write.mode("append").saveAsTable(f"{config['raw_catalog_name']}.{config['raw_schema_name']}.deliveries")

# Inventory table

inventory_df.write.mode("append").saveAsTable(f"{config['raw_catalog_name']}.{config['raw_schema_name']}.inventory")

# Orders table

orders_df.write.mode("append").saveAsTable(f"{config['raw_catalog_name']}.{config['raw_schema_name']}.orders")

In [0]:
deliveries_df.printSchema()

spark.sql("describe table zapflow_raw.default.deliveries").show()
# to c

In [0]:
#to c